### Step 1: Install necesscary packages

### Step 2: Package imports and configuration

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
import torch
import torch.nn.functional as F
import pickle
from model import GPT, GPTConfig
from tqdm import tqdm
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 15
batch_size = 64
max_length = 128
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [3]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [23]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

### Data generation explanation
My data generation pipeline was originally written in a separate file. To fit submission requirements, I have copy pasted the entire script inside the cell below.

The strategy was that, given the arithmetic requirement of the model was to train on math from a small range ~[0,100] I thought that it was tractable to procedurally generate all possible combinations of valid examples. I enforce validity of division examples by working backwards so that all quotients are integers.

How do I procedurally generate arithmetic expressions? First, the three possible examples per expression (arithmetic, 2 algebraic expressions with x in diff positions) only consists of x OP y and a result, so it was possible to convert 1 expression style to another, and in string CoT form the algebra to arithmetic expression only requires 1 reordering, so I could reuse arithmetic CoT string.

Simple COT reasoning: An autoregressive model attends to prior tokens so it made sense to emulate some form of chain-of-thought. Therefore a change I decided to make was to shift the sentinel string ("so the answer is Z") to the end of the expected response. In academic papers online (https://arxiv.org/html/2307.03381), there are examples that use longer CoT with the intuition to teach the model to actually try and do math with more primitive add and carry operations, but I tried my luck with a simple CoT first and it seemed to perform well enough that I did not need to modify the CoT style. Similarly, I thought perhaps more refusals beyond "Sorry, I don't know" like with off-by-one numbers in thinking might help the model be more accurate, but empirically I think my simple model performed satisfactorily to not require further extension.

With a little reorg and OOP, the dataset generator yields 92k examples, and there was already a shuffler defined.

I additionally added random sampler method to sample from the test set for evaluation.

Some notes:
- I didn't handle, for algebra transforms, double negatives, i.e. 12 -- 17 = ? However, because CoT was so simple the models eventually learnt that -- has the same effect as +.
- Because training data did not include negative integers [-100,100], when the first operand is below 0, it usually cannot solve the problem.

In [18]:
import json
import random
import dataclasses

LO, HI = 0, 100


def addition_generator():
    for a in range(LO, HI + 1):
        for b in range(LO, HI + 1):
            yield (a, b), a + b


def subtraction_generator():
    for a in range(LO, HI + 1):
        for b in range(LO, HI + 1):
            yield (a, b), a - b


def multiplication_generator():
    for a in range(LO, HI + 1):
        for b in range(LO, HI + 1):
            yield (a, b), a * b


def division_generator():
    # work backwards to generate only whole-number quotients
    for b in range(max(LO, 1), HI + 1):
        for q in range(0, HI // b + 1):
            a = b * q
            if LO <= a <= HI:
                yield (a, b), q


@dataclasses.dataclass(frozen=True)
class Problem:
    op: str                    # one of "+", "-", "*", "/"
    operands: tuple[int, int]  # (a, b); for algebra, x replaces one of them (see x_first)
    answer: int                # the arithmetic result a op b = answer (also the RHS of algebra)
    is_algebra: bool = False   # False: arithmetic, True: algebra
    x_first: bool = False      # algebra only: True -> x sits in the first position (x op b = ?)


    @property
    def solution(self) -> int:
        """The value the question actually asks for: the arithmetic answer,
        or the value of x (the hidden operand) for algebra."""
        if not self.is_algebra:
            return self.answer
        return self.operands[0] if self.x_first else self.operands[1]

    @property
    def question_string(self):
        a, b = self.operands
        if not self.is_algebra:
            return f"{a}{self.op}{b}=?"
        if self.x_first:
            return f"x{self.op}{b}={self.answer},x=?"
        return f"{a}{self.op}x={self.answer},x=?"


def make_arith_problem(op, generator) -> Problem | None:
    """Pull the next (operands, answer) tuple off `gen` and wrap it as a Problem.
    `gen` is a generator instance (e.g. addition_generator()); it remembers its
    own position, so each call materializes the next problem. Returns None once
    the generator is exhausted."""
    try:
        (a, b), answer = next(generator)
    except StopIteration:
        return None
    return Problem(op=op, operands=(a, b), answer=answer)


def algebra_from_arith(p: Problem, x_first: bool) -> Problem | None:
    """
    Turn a valid arithmetic problem into its algebra version by hiding one
    operand (x_first decides which), so a whole-number answer is guaranteed

    Return None when x would not be uniquely determined:
      - x * 0 = 0       (any x works)
      - 0 / x = 0       (any nonzero x works)
      - x / b = 0       (any x in 0..b-1 works, when b > 1)
    """
    a, b = p.operands
    if p.op == "*" and (a == 0 or b == 0):
        return None  # x * 0 = 0 permits any x
    if p.op == "/" and (b == 0 or p.answer == 0):
        # answer == 0: zero quotient -> x/b = 0 or 0/x = 0, both non-unique.
        return None
    return dataclasses.replace(p, is_algebra=True, x_first=x_first)


#### Engine to compose a CoT and wrap it with the answer sentinel.

def arithmetic_correct_cot(p: Problem) -> str:
    """Short correct CoT for an arithmetic problem, e.g. 'A+B equals C'."""
    a, b = p.operands
    return f"{a}{p.op}{b} equals {p.answer}"


def algebra_to_arithmetic(p: Problem) -> Problem:
    """Rewrite an algebra problem with the inverse operation, as an arith
    Problem (x-18=21 -> 21+18; x/10=6 -> 6*10; a-x=c -> a-c; ...)."""
    a, b = p.operands
    c = p.answer
    if p.x_first:
        inv_op = {"+": "-", "-": "+", "*": "/", "/": "*"}[p.op]
        return Problem(op=inv_op, operands=(c, b), answer=a)
    if p.op in ("+", "*"):
        inv_op = "-" if p.op == "+" else "/"
        return Problem(op=inv_op, operands=(c, a), answer=b)
    # for minus and divide can just swap operands
    # e.g. 11-x=6 -> 11-6=x
    return Problem(op=p.op, operands=(a, c), answer=b)


def algebraic_correct_cot(p: Problem) -> str:
    """Reduce via algebra_to_arithmetic(), then reuse arithmetic_correct_cot()."""
    arith_p = algebra_to_arithmetic(p)
    a, b = arith_p.operands
    return f"x={a}{arith_p.op}{b}, and {arithmetic_correct_cot(arith_p)}"


def wrap_with_sentinel(answer: object, cot: str, front: bool = False) -> str:
    cot = cot.replace(".", ",")  # Assign back: str.replace() returns a new string
    if front:
        return f"The answer is {answer} because {cot}."
    return f"{cot}, so the answer is {answer}."


def correct_composer(p: Problem, front: bool = False) -> str:
    if p.is_algebra:
        cot = algebraic_correct_cot(p)
    else:
        cot = arithmetic_correct_cot(p)
    return wrap_with_sentinel(p.solution, cot, front)


def wrong_refusal(p: Problem) -> str:
    del p  # consume p
    return "Sorry, I do not know."


def sample_negatives(p, k) -> list[str]:
    """
    no duplicate negatives for the same problem
    """
    # For now, set refusal just to be hardcoded
    del k
    return [wrong_refusal(p)]

Pair = dict[str, str]

def build_pairs(p: Problem, k: int = 1, front: bool = False) -> list[Pair]:
    prompt = p.question_string
    positive_response = f"{prompt} {correct_composer(p, front=front)}"
    negative_responses = sample_negatives(p, k)
    pairs = []
    for neg in negative_responses:
        pairs.append(
            {
                "positive": positive_response,
                "negative": f"{prompt} {neg}",
            }
        )
    return pairs


def write_pairs(pairs: list[dict], path: str) -> None:
    with open(path, "w") as f:
        json.dump(pairs, f, indent=2)


def iter_all_problems():
    generators = [
        ("+", addition_generator()),
        ("-", subtraction_generator()),
        ("*", multiplication_generator()),
        ("/", division_generator()),
    ]
    for op, gen in generators:
        p = make_arith_problem(op, gen)
        while p is not None:
            yield p
            for x_first in (True, False):
                alg_p = algebra_from_arith(p, x_first=x_first)
                if alg_p is not None:
                    yield alg_p
            p = make_arith_problem(op, gen)

def generate_random_problems(n: int) -> list[Problem]:
    """Returns a random sample of n problems."""
    all_problems = list(iter_all_problems())
    return random.sample(all_problems, n)

def main_loop():
    output_path = "pos_neg_pairs.json"
    k = 1
    front = False
    pairs: list[Pair] = []
    for p in iter_all_problems():
        pairs.extend(build_pairs(p, k=k, front=front))
    write_pairs(pairs, output_path)

main_loop()

In [19]:

data_path = "./pos_neg_pairs.json"
with open(data_path, "r", encoding="utf-8") as f:
    lines = json.load(f)


In [20]:
len(lines)

92953

### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

I use the recommended adamw optimizer and cosine annealing for the learning rate applied with some sane defaults.

In [ ]:
# recommend to use the AdamW optimizer
optim = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)
total_steps = (len(lines) // batch_size) * epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=total_steps, eta_min=base_lr*0.1)

### Step 7: Begin training (**students are required to complete this part!**)

since the code skeleton was there and the helper functions to calculate logprobs were there already, I just needed to add the usual boilerplate code (zero the gradients, do backprop, clipping grad norms to prevent explosion, step optim and scheduler etc).

I just used the loss provided in the notebook even though it's a little different from the one in the DPO paper.

Some differences in my initial hparam config:
- max_length increased to support COT
- increased # of epochs since final model performance was positively affected by increasing # of epochs.

In [ ]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size), total=total_steps)
    for step, (neg_tensor, pos_tensor) in enumerate(pbar):
        optim.zero_grad()
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        optim.step()
        scheduler.step()
        pbar.set_description(f"Epoch {epoch + 1}/{epochs} | Loss: {loss.item():.4f}")
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

Epoch 1/15 | Loss: 0.0282: 100%|██████████| 1452/1452 [01:01<00:00, 23.63it/s]


Saved checkpoint to ./dpo.pt


Epoch 2/15 | Loss: 0.0261: 100%|██████████| 1452/1452 [01:01<00:00, 23.61it/s]


Saved checkpoint to ./dpo.pt


Epoch 3/15 | Loss: 0.0228: 100%|██████████| 1452/1452 [01:01<00:00, 23.59it/s]


...

Saved checkpoint to ./dpo.pt


Epoch 14/15 | Loss: 0.0186: 100%|██████████| 1452/1452 [01:01<00:00, 23.61it/s]


Saved checkpoint to ./dpo.pt


Epoch 15/15 | Loss: 0.0184: 100%|██████████| 1452/1452 [01:01<00:00, 23.60it/s]

Saved checkpoint to ./dpo.pt


### Step 8: Begin testing (**students are required to complete this part!**)

For my evaluation pipeline, I just defined a simple answer matcher.

The model learns to output every answer with "...so the answer is X.", and the period causes generation to terminate, allowing a straightforward string match.

I pull a random sample of 500 problems to test on the model and tracked the model's accuracy. By construction each Problem object already has the correct answer as an attribute.

In [21]:
# string match
def extract_answer(s):
    # find "so the answer is {ans}." and return ans
    return s.split("so the answer is ")[-1].split(".")[0].strip()


test_set_problems: list[Problem] = generate_random_problems(500)

In [22]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()

correct_responses = []
wrong_responses = []

with torch.no_grad():
    for problem in test_set_problems:
        prompt = problem.question_string
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
        y = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        resp = decode(y[0].squeeze().tolist())
        if extract_answer(resp) == str(problem.solution):
            # uncomment if you want to see correct responses
            # print(f"Correct: {resp}")
            correct_responses.append(resp)
        else:
            print(resp)
            wrong_responses.append(resp)

print(f"Correct: {len(correct_responses)}, Wrong: {len(wrong_responses)}, Accuracy: {len(correct_responses) / (len(correct_responses) + len(wrong_responses)):.4f}")


5*97=? 5*97 equals 495, so the answer is 495.
94*66=? 94*66 equals 6196, so the answer is 6196.
40*62=? 40*62 equals 2520, so the answer is 2520.
56*x=1848,x=? x=1848/56, and 1848/56 equals 38, so the answer is 38.
21*59=? 21*59 equals 1389, so the answer is 1389.
34*54=? 34*54 equals 1916, so the answer is 1916.
49*25=? 49*25 equals 1145, so the answer is 1145.
88*77=? 88*77 equals 6446, so the answer is 6446.
85*19=? 85*19 equals 1675, so the answer is 1675.
57*x=4047,x=? x=4047/57, and 4047/57 equals 61, so the answer is 61.
x*45=4275,x=? x=4275/45, and 4275/45 equals 97, so the answer is 97.
61*47=? 61*47 equals 2917, so the answer is 2917.
70*24=? 70*24 equals 1780, so the answer is 1780.
x*68=5440,x=? x=5440/68, and 5440/68 equals 85, so the answer is 85.
18*98=? 18*98 equals 1776, so the answer is 1776.
21*41=? 21*41 equals 831, so the answer is 831.
x*32=2848,x=? x=2848/32, and 2848/32 equals 84, so the answer is 84.
41*63=? 41*63 equals 2793, so the answer is 2793.
31*16=? 31*

The model doesn't respond well to out of distribution prompts.

The model likely cannot generalize math without including these other types of expressions in the data.

However, I suspect the model is essentially performing memorizing, since the entire space of X OP Y expressions was training data. I am essentially training on a test set.

With enough compute, I'd likely procedurally generate a base of these examples and probably also train with CoT artifacts that might actually teach the model how to generalize.

In [ ]:
weird_prompts = [
    "5*-4=?", # negative operands
    "-4-5=?",
    "-2=?",  # 1 operand
    "4=?",
    "x=4, x=?",
    "132+2=?", # operand > 100.
]

for prompt in weird_prompts:
    with torch.no_grad():
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
        y = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        resp = decode(y[0].squeeze().tolist())
        print(resp)

5*-4=? 5*44 equals 220, so the answer is 220.
-4-5=? -5 equals -1, so the answer is -1.
-2=? -2, and -2+2 equals -2, so the answer is -2.
4=? 4*4 equals 196, so the answer is 196.
x=4, x=? x=4*4, and 4*4 equals 16, so the answer is 6.
132+2=? 13+22 equals 35, so the answer is 35.


In [8]:
# review responses
correct_responses

['80+x=164,x=? x=164-80, and 164-80 equals 84, so the answer is 84.',
 '15+16=? 15+16 equals 31, so the answer is 31.',
 '30+x=99,x=? x=99-30, and 99-30 equals 69, so the answer is 69.',
 'x-49=13,x=? x=13+49, and 13+49 equals 62, so the answer is 62.',
 'x+60=115,x=? x=115-60, and 115-60 equals 55, so the answer is 55.',
 '18-x=-9,x=? x=18--9, and 18--9 equals 27, so the answer is 27.',
 '81-x=-2,x=? x=81--2, and 81--2 equals 83, so the answer is 83.',
 '53-30=? 53-30 equals 23, so the answer is 23.',
 'x-11=58,x=? x=58+11, and 58+11 equals 69, so the answer is 69.',
 '8-67=? 8-67 equals -59, so the answer is -59.',
 '48+19=? 48+19 equals 67, so the answer is 67.',
 '77-x=-8,x=? x=77--8, and 77--8 equals 85, so the answer is 85.',
 '93+x=106,x=? x=106-93, and 106-93 equals 13, so the answer is 13.',
 'x+34=75,x=? x=75-34, and 75-34 equals 41, so the answer is 41.',
 '42+x=114,x=? x=114-42, and 114-42 equals 72, so the answer is 72.',
 '13-100=? 13-100 equals -87, so the answer is -87.